In [13]:
import numpy as np
import pandas as pd
import tensorflow as tf
import random
from tensorflow.keras.models import Sequential, clone_model
from tensorflow.keras.layers import LSTM, Dropout, Dense
from tensorflow.keras.optimizers import Adam, SGD
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.metrics import BinaryAccuracy, AUC
from tensorflow.keras.preprocessing.sequence import TimeseriesGenerator
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report, roc_curve, auc


In [14]:
# Set seed for reproducibility
seed = 42
np.random.seed(seed)
tf.random.set_seed(seed)
random.seed(seed)

In [15]:
# Load the dataset
data_x = pd.read_csv('X_cma.csv')
data_y = pd.read_csv('y_cma.csv')

In [16]:
# Remove the 'Unnamed: 0' column if it exists
if 'Unnamed: 0' in data_x.columns:
    data_x = data_x.drop(columns=['Unnamed: 0'])
if 'Unnamed: 0' in data_y.columns:
    data_y = data_y.drop(columns=['Unnamed: 0'])

In [17]:
# Convert to numpy arrays
X = data_x.values
y = data_y['Target'].values

In [18]:
# Define sliding window parameters
window_size = 600
test_gap = 180
step_size = 1

In [19]:
# Lists to store results
all_test_predictions = []
all_y_pred_results = []
all_y_true_results = []
all_confusion_metrics = []
all_classification_reports = []

In [20]:
# Define model creation function
def create_model(hu=20, lookback=10, features=1):
    tf.keras.backend.clear_session()
    model = Sequential()
    model.add(LSTM(units=hu, input_shape=(lookback, features), activation='relu', return_sequences=False, name='LSTM1'))
    model.add(Dropout(0.4, name='Dropout1'))
    model.add(Dense(units=1, activation='sigmoid', name='Output'))
    opt = Adam(learning_rate=0.001, epsilon=1e-08, decay=0.0)
    model.compile(optimizer=opt, loss=BinaryCrossentropy(), metrics=[BinaryAccuracy(name='accuracy', threshold=0.5), AUC()])
    return model

In [21]:
# Initialize and save the initial model
initial_model = create_model(hu=20, lookback=10, features=X.shape[1])
initial_model.compile(optimizer=Adam(learning_rate=0.001, epsilon=1e-08, decay=0.0), loss=BinaryCrossentropy(), metrics=[BinaryAccuracy(name='accuracy', threshold=0.5), AUC()])
initial_model.save('initial_model.h5')

c:\Users\tonfo\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\rnn\rnn.py:204: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
c:\Users\tonfo\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\optimizers\base_optimizer.py:33: UserWarning: Argument `decay` is no longer supported and will be ignored.
  warnings.warn(


In [22]:
# Save the weights of the initial model
initial_model.save_weights('current_model.weights.h5')

In [23]:
# EarlyStopping callback
early_stopping = EarlyStopping(monitor='loss', patience=10, restore_best_weights=True)

In [24]:
# Train and evaluate the model with sliding window
for start in range(0, len(X) - window_size - test_gap, step_size):
    end = start + window_size
    X_train_window = X[start:end]
    y_train_window = y[start:end]
    
    # Define test indices
    test_start = end + test_gap
    test_end = test_start + 1
    if test_end > len(X):
        break
    X_test_window = X[test_start:test_end]
    y_test_window = y[test_start:test_end]

    # Sequence length for training
    seqlen = 10
    # Number of features
    numfeat = X_train_window.shape[1]

    # Generate sequence input for the training dataset
    train_generator = TimeseriesGenerator(X_train_window, y_train_window, length=seqlen, batch_size=32)

    # Load and clone the initial model for the first iteration, then the previously trained model for subsequent iterations
    model = create_model(hu=20, lookback=10, features=numfeat)
    model.load_weights('current_model.weights.h5')

    model_clone = clone_model(model)
    model_clone.set_weights(model.get_weights())

    model_transfer = Sequential(model_clone.layers[:-1])
    model_transfer.add(Dense(1, activation='sigmoid'))

    # Freeze the cloned layers initially
    for layer in model_transfer.layers[:-1]:
        layer.trainable = False

    model_transfer.compile(loss=BinaryCrossentropy(), optimizer=SGD(learning_rate=1e-3), metrics=[BinaryAccuracy(name='accuracy', threshold=0.5), AUC()])
    model_transfer.fit(train_generator, epochs=4, verbose=1, class_weight=dict(enumerate(compute_class_weight('balanced', classes=np.unique(y_train_window), y=y_train_window))), shuffle=False, callbacks=[early_stopping])

    # Unfreeze the cloned layers for further training
    for layer in model_transfer.layers[:-1]:
        layer.trainable = True

    model_transfer.compile(loss=BinaryCrossentropy(), optimizer=SGD(learning_rate=1e-3), metrics=[BinaryAccuracy(name='accuracy', threshold=0.5), AUC()])
    model_transfer.fit(train_generator, epochs=70, verbose=1, class_weight=dict(enumerate(compute_class_weight('balanced', classes=np.unique(y_train_window), y=y_train_window))), shuffle=False, callbacks=[early_stopping])

    # Save the best model during training for the current window
    model_transfer.save('current_model.h5')
    model_transfer.save_weights('current_model.weights.h5')

    # Predict on the single test sample directly
    test_input = np.expand_dims(X_test_window, axis=0)
    prediction = model_transfer.predict(test_input)
    print(f'Test Prediction: {prediction[0][0]:.2f}')

    # Evaluate with the training set
    y_true_train = y_train_window[-(len(X_train_window)-seqlen):]
    y_pred_train_probs = model_transfer.predict(train_generator)
    y_pred_train = (y_pred_train_probs > 0.5).astype(int)

    # Calculate evaluation metrics
    train_accuracy = accuracy_score(y_true_train, y_pred_train)
    train_precision = precision_score(y_true_train, y_pred_train)
    train_recall = recall_score(y_true_train, y_pred_train)
    train_f1 = f1_score(y_true_train, y_pred_train)
    train_roc_auc = roc_auc_score(y_true_train, y_pred_train_probs)

    print(f'Training Accuracy: {train_accuracy:.2f}')
    print(f'Training Precision: {train_precision:.2f}')
    print(f'Training Recall: {train_recall:.2f}')
    print(f'Training F1 Score: {train_f1:.2f}')
    print(f'Training ROC AUC: {train_roc_auc:.2f}')

    # Store results
    all_test_predictions.append(prediction[0][0])
    all_y_pred_results.extend(y_pred_train.tolist())
    all_y_true_results.extend(y_true_train.tolist())
    all_confusion_metrics.extend(confusion_matrix(y_true_train, y_pred_train).ravel().tolist())
    all_classification_reports.append(classification_report(y_true_train, y_pred_train, output_dict=True))

    # Evaluate with the test set
    y_true_test = y_test_window
    y_pred_test_probs = prediction  # Use prediction directly as y_pred_probs

    # Binary classification on test set
    y_pred_test_cm = (y_pred_test_probs > 0.5).astype(int)

    # Print classification report for test set
    print("Classification Report for Test Set")
    print(classification_report(y_true_test, y_pred_test_cm))

    # Determine confusion matrix for test set
    cm_test = confusion_matrix(y_true_test, y_pred_test_cm)
    if cm_test.shape == (1, 1):
        tn, fp, fn, tp = cm_test[0, 0], 0, 0, 0
    elif cm_test.shape == (2, 2):
        tn, fp, fn, tp = cm_test.ravel()
    else:
        raise ValueError("Unexpected confusion matrix shape")

    # Store results for the test set
    all_confusion_metrics.extend([tn, fp, fn, tp])
    all_classification_reports.append(classification_report(y_true_test, y_pred_test_cm, output_dict=True))

    # Determine AUC for the test set
    fpr_test, tpr_test, thresholds_test = roc_curve(y_true_test, y_pred_test_probs)
    roc_auc_test = auc(fpr_test, tpr_test)

Epoch 1/4
19/19 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7899 - auc_1: 0.6677 - loss: 4.8401    
Epoch 2/4
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.5758 - auc_1: 0.5178 - loss: 20.8456     
Epoch 3/4
 1/19 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - accuracy: 0.0938 - auc_1: 0.0000e+00 - loss: 40.9985

c:\Users\tonfo\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.5741 - auc_1: 0.5328 - loss: 19.7288     
Epoch 4/4
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.5839 - auc_1: 0.5500 - loss: 17.8001     
Epoch 1/70
19/19 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.3486 - auc_2: 0.3940 - loss: 5.0877        
Epoch 2/70
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.3386 - auc_2: 0.4750 - loss: 0.9511         
Epoch 3/70
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.2748 - auc_2: 0.4750 - loss: 0.9512       
Epoch 4/70
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.2960 - auc_2: 0.4750 - loss: 0.9514         
Epoch 5/70
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.2553 - auc_2: 0.4750 - loss: 0.9515         
Epoch 6/70
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.2845 - auc_2: 0.4750 - loss: 0.9516       
Epoch 7/70
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.3182 - auc_2: 0.4750 - loss: 0.9517         
Epoch 8/70
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 3

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 115ms/step
Test Prediction: 0.51
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step
Training Accuracy: 0.77
Training Precision: 0.77
Training Recall: 1.00
Training F1 Score: 0.87
Training ROC AUC: 0.46
Classification Report for Test Set
              precision    recall  f1-score   support

           1       1.00      1.00      1.00         1

    accuracy                           1.00         1
   macro avg       1.00      1.00      1.00         1
weighted avg       1.00      1.00      1.00         1



c:\Users\tonfo\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1517: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\tonfo\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1517: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\tonfo\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1517: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, mo

Epoch 1/4


... (5,159 outputs left out)

19/19 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.5202 - auc_1: 0.4000 - loss: 0.7097        
Epoch 2/4
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6792 - auc_1: 0.4000 - loss: 0.7097   
Epoch 3/4


c:\Users\tonfo\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6792 - auc_1: 0.4000 - loss: 0.7096     
Epoch 4/4
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6792 - auc_1: 0.4000 - loss: 0.7096   
Epoch 1/70
19/19 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.6792 - auc_2: 0.4000 - loss: 0.7097     
Epoch 2/70
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6792 - auc_2: 0.4000 - loss: 0.7096   
Epoch 3/70
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6792 - auc_2: 0.4000 - loss: 0.7096   
Epoch 4/70
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6816 - auc_2: 0.4000 - loss: 0.7096   
Epoch 5/70
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6640 - auc_2: 0.4000 - loss: 0.7096   
Epoch 6/70
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6640 - auc_2: 0.4000 - loss: 0.7095   
Epoch 7/70
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6640 - auc_2: 0.4000 - loss: 0.7095   
Epoch 8/70
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6588 - auc_2: 0.4

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 139ms/step
Test Prediction: 0.50
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step


In [1]:
# Save the final model
model_transfer.save('final_model.h5')

NameError: name 'model_transfer' is not defined

In [ ]:
# Save the results to files for further analysis
results = {
    "test_predictions": all_test_predictions,
    "y_pred_results": all_y_pred_results,
    "y_true_results": all_y_true_results,
    "confusion_metrics": all_confusion_metrics,
    "classification_reports": all_classification_reports
}

In [ ]:
results_df = pd.DataFrame(results)
results_df.to_csv('CMA_results.csv', index=False)